While 'notebook\data_ingestion.ipynb' acts as our high-recall gatekeeper (generating candidate pairs),

```

'notebook\feature_tuning.ipynb' (this notebook) is where we optimize our pipeline for the precision-heavy [F_{0.5}] metric.

```

This is the exact playground where you build your feature store and fine-tune your model thresholds before exporting our final script package.

Its Core Engineering Jobs:

```

## Algorithmic Distance Extraction:

It reads your filtered candidate_pairs.tsv file. It runs multi-threaded RapidFuzz and jellyfish functions to compute Jaro-Winkler scores, token sort ratios, and phonetic alignments across your column pairs.

## Local Validation Diagnostics:

It loads your 20% holdout split (the data you isolated to simulate the unseen test dataset) to evaluate model performance without data leakage.

## Optimizing the \[F_{0.5}\] Threshold:

Because a false merge (matching two different businesses) heavily penalizes your score, you use this notebook to dynamically test classification boundaries (e.g., boundaries from 0.80 to 0.95). You find the exact point that maximizes precision, protects your singletons, and secures the highest possible metric score.

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import precision_recall_fscore_support

def calculate_macro_f05(precision, recall):
    """Calculates the precision-heavy F_0.5 metric specified by the challenge."""
    if (0.25 * precision + recall) == 0:
        return 0.0
    return (1.25 * precision * recall) / (0.25 * precision + recall)

print("-> Initializing Threshold Tuning Simulator for F_0.5 Mastery...")

# 1. Load your local validation dense feature matrix (Generated via RapidFuzz)
# For prototyping, we simulate a small evaluation matrix below
simulated_data = {
    'feat_name_jw': [0.95, 0.88, 0.42, 0.91, 0.78],
    'feat_name_tsr': [0.98, 0.82, 0.35, 0.89, 0.65],
    'feat_addr_lev': [0.92, 0.75, 0.21, 0.88, 0.54],
    'feat_phonetic': [1.0, 1.0, 0.0, 1.0, 0.0],
    'feat_country': [1.0, 1.0, 1.0, 1.0, 1.0],
    'true_label': [1, 1, 0, 1, 0] # Ground truth matching labels
}
df_val = pd.DataFrame(simulated_data)
feature_cols = ["feat_name_jw", "feat_name_tsr", "feat_addr_lev", "feat_phonetic", "feat_country"]

# 2. Load your pre-trained LightGBM model binary
# (In production, this points to your frozen model file)
# bst = lgb.Booster(model_file="../Package/code/business_entity_resolution/src/models/final_lightgbm_matcher.txt")

# For simulation, we compute a simple weighted probability score
df_val['prob'] = (df_val['feat_name_jw'] * 0.4) + (df_val['feat_addr_lev'] * 0.4) + (df_val['feat_phonetic'] * 0.2)

# 3. Dynamic Sweep: Test multiple precision boundaries to find the peak score
thresholds = np.arange(0.70, 0.95, 0.05)
best_threshold = 0.85
max_f05 = 0.0

print("\n--- Running Precision Boundary Optimization Search ---")
for t in thresholds:
    # Convert probabilities to binary predictions based on current threshold
    predictions = (df_val['prob'] >= t).astype(int)
    
    # Extract standard metrics
    precision, recall, _, _ = precision_recall_fscore_support(df_val['true_label'], predictions, average='binary', zero_division=0)
    
    # Calculate the challenge-specific precision-heavy macro score
    f05 = calculate_macro_f05(precision, recall)
    print(f"Threshold: {t:.2f} | Precision: {precision:.3f} | Recall: {recall:.3f} | F_0.5 Score: {f05:.3f}")
    
    if f05 > max_f05:
        max_f05 = f05
        best_threshold = t

print("="*66)
print(f"\n Optimization Matrix Complete!")
print("\n")
print("="*66)
print("\n")
print(f"Peak F_0.5 Performance: {max_f05:.3f} achieved at strict threshold: {best_threshold:.2f}")
print(f"-> Action: Hardcode {best_threshold:.2f} as HIGH_PRECISION_THRESHOLD inside classifier.py.")
print("\n")
print("="*66)
